treetime
treetimeを用いて最も古い=祖先に近いと考えられる配列でrerootしてみる。
inputにtree, alignment, dates.csv
collection dateがmetaにある配列の切り出しとmetaの作成

In [2]:
import pandas as pd
from Bio import SeqIO, Entrez
from Bio.Seq import Seq 
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
mpl.rcParams['svg.fonttype'] = 'none'
# Path to your Arial.ttf file
# arial_font_path = '/home/skawakubo/fonts/Arial.ttf'
# Create a FontProperties object for the Arial font
# arial_font = FontProperties(fname=arial_font_path)
# mpl.rcParams['font.family'] = arial_font.get_name()
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import subprocess

In [3]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [4]:
# read fasta
FCoV1_S_full_seq_path = '/Users/izumikida/FCoV_seq_relatives/key_mut_detection/Spike/FCoV1_S_all_seq_cleaned_cds_linsi_curated_fullllength_fix.fasta'
FCoV1_S_full_seq = fasta_to_dataframe(FCoV1_S_full_seq_path)
FCoV1_S_full_seq.rename(columns={"ID": "Accession"}, inplace=True)
FCoV1_S_full_seq

,Accession,seq
0,PQ416775.1,atgatagtgttaatacttgcacttcttagcaccgccagctctgaag...
1,PX315748.1,atgatagtgctaatacttgcattcgtaagtgtcgttagttctaacg...
2,PX315749.1,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
3,PX315750.1,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
4,PX315751.1,atgatagtgttaatacttgcactccttagctttgctaggtctgacg...
...,...,...
119,AB088222.1,atgatattgttaatatttgcactctttagcgttgttagatctgaag...
120,FJ917522.1,atgatattgttactactggcaattcttagcgccgttaaggctagag...
121,FJ917535.1,atgatattgttactactggcaattcttagcgccgttaaggctagag...
122,FJ917523.1,atgatattgttactactggcaattcttagcgccgttaaggctagag...


In [5]:
meta = pd.read_csv("/Users/izumikida/FCoV_seq_relatives/FCoV1_S_final/FCoV1_S_all_seq_final_meta_20260306.tsv", sep="\t")
meta

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI
...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,JQ304330.1,FECV,Feline coronavirus,FECV141,taxon:12663,348,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI
444,FJ917532.1,FIPV,Feline coronavirus,FIPV-UCD11b-1a,taxon:12663,186,NaN,USA,Felis catus (cat),diseased lesional tissue,31-Mar-2008,24-JUL-2016,NCBI
445,PP097359.1,FIPV,Feline coronavirus,FIPV/UGM/BUDF,taxon:12663,360,NaN,Indonesia,Felis catus,ascites,NaN,04-FEB-2024,NCBI
446,JQ304361.1,FECV,Feline coronavirus,FECV378,taxon:12663,543,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI


In [6]:
# Merge the metadata and the sequence
FCoV1_S = pd.merge(meta, FCoV1_S_full_seq, on="Accession")
FCoV1_S

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,seq
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,atgatagtgttaatacttgcacttcttagcaccgccagctctgaag...
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,atgatagtgctaatacttgcattcgtaagtgtcgttagttctaacg...
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,atgatagtgttaatacttgcactccttagctttgctaggtctgacg...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,AB088222.1,FIPV,Feline infectious peritonitis virus,NaN,taxon:11135,4374,NaN,NaN,NaN,NaN,NaN,26-JUN-2008,NCBI,atgatattgttaatatttgcactctttagcgttgttagatctgaag...
120,FJ917522.1,FECV,Feline coronavirus,FECV-UCD5,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,12-Feb-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...
121,FJ917535.1,FIPV,Feline coronavirus,FIPV-UCD11b-2b,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,04-Apr-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...
122,FJ917523.1,FIPV,Feline coronavirus,FIPV-UCD13,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,17-Jan-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...


In [27]:
FCoV1_S_full_date = FCoV1_S[["Accession", "Collection_Date"]]

In [20]:
FCoV1_S_full_date.to_csv('FCoV1_S_full_prelim_date.tsv', sep='\t', index=False)

## UCDの3検体のみ2024と推定した上でここまでdataの形式をyear-month-dateに修正、不明なものは01と入力
3つdate不明なものとoutlierのc57を以下で削除する。

In [12]:
list_rm = ['DQ848678.1','D32044.1','AB088222.1', 'c57']

In [13]:
meta2 = pd.read_csv("/Users/izumikida/FCoV_seq_relatives/treetime/NRR_dates/FCoV1_S_full_prelim_fixed_date.tsv", sep="\t")

In [14]:
meta2_filtered = meta2[~meta2["Accession"].isin(list＿rm)].copy()
meta2_filtered

,Accession,Collection_Date
0,PQ416775.1,2023-11-05
1,PX315748.1,2011-10-01
2,PX315749.1,2013-01-11
3,PX315750.1,2013-01-11
4,PX315751.1,2013-06-07
...,...,...
115,FJ917521.1,2008-02-12
116,FJ917522.1,2008-02-12
117,FJ917535.1,2008-04-04
118,FJ917523.1,2008-01-17


In [15]:
# 指定したAccessionを "含まない" (~isin) 行だけを抽出
FCoV1_S_filtered = FCoV1_S[~FCoV1_S["Accession"].isin(list_rm)].copy()
FCoV1_S_filtered

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,seq
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,atgatagtgttaatacttgcacttcttagcaccgccagctctgaag...
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,atgatagtgctaatacttgcattcgtaagtgtcgttagttctaacg...
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,atgatagtgctaatacttgcattcgtaagtgttgttagttctcacg...
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,atgatagtgttaatacttgcactccttagctttgctaggtctgacg...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,FJ917521.1,FIPV,Feline coronavirus,FIPV-UCD12,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,12-Feb-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...
120,FJ917522.1,FECV,Feline coronavirus,FECV-UCD5,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,12-Feb-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...
121,FJ917535.1,FIPV,Feline coronavirus,FIPV-UCD11b-2b,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,04-Apr-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...
122,FJ917523.1,FIPV,Feline coronavirus,FIPV-UCD13,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,17-Jan-2008,24-JUL-2016,NCBI,atgatattgttactactggcaattcttagcgccgttaaggctagag...


In [16]:
# write out as fasta file
with open('FCoV1_S_full_date_treetime_0313.fasta', 'w') as f:
    for index, row in FCoV1_S_filtered.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")

c57がoutlierとなるため削除した。fixed_date.tsvでも編集済み。
NRR_dates_treeに最終versionのdates情報のない3seqとc57を除いた120seqのNRRごとのfastaを格納。